In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F

emp_data = spark.table("organization.employee_events")
department = spark.table("organization.department")
pipeline_run = spark.table("organization.pipeline_runs")


def normalize_df(df,partition_by, order_by):
    windows = Window.partitionBy(partition_by).orderBy(F.col(order_by).desc())
    normalized = df.withColumn("rowNumber", F.row_number().over(windows))\
                .where(F.col("rowNumber") == 1)\
                .where(F.col("event_type") != "TERMINATED")\
                .drop("rowNumber")
    return normalized
def history(df, lag_col, partition_col_name,order_col_name,default_val, new_col_name):
    window = Window.partitionBy(partition_col_name).orderBy(F.col(order_col_name))
    history_df = df.withColumn(new_col_name, F.lag(F.col(lag_col), default = default_val).over(window))
    return history_df
def lead(df, lead_col, partition_col_name,order_col_name,default_val, new_col_name):
    window = Window.partitionBy(partition_col_name).orderBy(F.col(order_col_name))
    lead_df = df.withColumn(new_col_name, F.lead(F.col(lead_col), default = default_val).over(window))
    return lead_df
def date_diff(df, start_date, end_date, new_col_name):
    new_df = df.withColumn(new_col_name, F.datediff(F.col(start_date), F.col(end_date)))
    return new_df
def val_diff(df, val_1, val_2, new_col_name):
    new_df = df.withColumn(new_col_name, df[val_1]- df[val_2])
    return new_df
default_timestamp = None
latest_emp_details = normalize_df(emp_data,"emp_id","even_ts")
emp_event_history = history(emp_data,"event_type", "emp_id","even_ts","ON-BORDING","Event_history")
emp_effective_from = history(emp_data,"even_ts", "emp_id", "even_ts",default_timestamp, "effective_from")
emp_ts = lead(emp_effective_from,"even_ts","emp_id", "even_ts", default_timestamp, "effective_to")
emp_previous_salary = history(emp_data,"salary","emp_id", "even_ts", 0, "salary_history")
emp_event_happening_time = date_diff(emp_ts, "effective_to", "effective_from", "time_for_next_event")
emp_salary_diff = val_diff(emp_previous_salary, "salary","salary_history", "salary_difference")

emp_salary_validity_check = emp_salary_diff.withColumn("validity_check", F.when(F.col("salary_difference") < 0, "INVALID")\
                                                                         .when(F.col("salary_difference") == 0, "NEW JOINER")\
                                                                         .when(F.col("salary_difference").isNull(), "NO CHANGE")\
                                                                         .otherwise("VALID")
                                                                         ).orderBy(F.col("even_ts"))
                                                                         

# emp_data.show()
# latest_emp_details.show()
# emp_event_history.show()
# emp_ts.show()
emp_event_happening_time.show()
emp_salary_diff.show()
emp_salary_validity_check.show()
None